# PCL Data Augmentation

This notebook runs the augmentation pipeline for Subtask 1 and saves the generated rows to a CSV.

Output artifact:
- `augmented_train_data.csv` (or `/kaggle/working/augmented_train_data.csv` on Kaggle)

Use the companion training notebook to load this CSV and append it to the original training split.


In [1]:
!pip install -q huggingface_hub nlpaug sacremoses google-genai nltk contractions python-dotenv contractions


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00


In [2]:
import os
import re
import random
import time

import numpy as np
import pandas as pd
import torch

import nltk
from nltk.corpus import wordnet, stopwords as nltk_stopwords

import nlpaug.augmenter.word as naw

from google import genai
from google.genai import types as genai_types

for _res in ['wordnet', 'stopwords', 'punkt', 'omw-1.4', 'averaged_perceptron_tagger']:
    nltk.download(_res, quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


In [3]:
# Optional Hugging Face login (safe to skip if not needed)
from huggingface_hub import login
KAGGLE = True
if KAGGLE:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    GEMINI_API_KEY = user_secrets.get_secret('GEMINI_API_KEY')
    if hf_token:
        login(token=hf_token)
        print('HF token loaded from .env and login completed.')
else:
    from dotenv import load_dotenv
    
    load_dotenv()
    hf_token = os.getenv('HF_TOKEN')
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
    if hf_token:
        login(token=hf_token)
        print('HF token loaded from .env and login completed.')




HF token loaded from .env and login completed.


In [4]:
# ============================================================
# Configuration
# ============================================================

# Choose which split(s) to augment.
# Set AUGMENT_TRAIN_SPLIT=False if train augmentations were already generated.
AUGMENT_TRAIN_SPLIT = False
AUGMENT_DEV_SPLIT = True

if not (AUGMENT_TRAIN_SPLIT or AUGMENT_DEV_SPLIT):
    raise ValueError('At least one of AUGMENT_TRAIN_SPLIT or AUGMENT_DEV_SPLIT must be True.')

# Augmentation controls (per original PCL sample)
SYNONYM_COPIES_PER_SAMPLE = 1
BACK_TRANS_COPIES_PER_SAMPLE = 2
GEMINI_COPIES_PER_SAMPLE = 2
DUPLICATE_COPIES_PER_SAMPLE = 2

TOTAL_NEW_PER_SAMPLE = (
    SYNONYM_COPIES_PER_SAMPLE
    + BACK_TRANS_COPIES_PER_SAMPLE
    + GEMINI_COPIES_PER_SAMPLE
    + DUPLICATE_COPIES_PER_SAMPLE
)

if TOTAL_NEW_PER_SAMPLE != 7:
    raise ValueError(f'Expected TOTAL_NEW_PER_SAMPLE=7, got {TOTAL_NEW_PER_SAMPLE}')

SYNONYM_N_MIN = 1            # min words replaced per sentence (WordNet)
SYNONYM_N_MAX = 3            # max words replaced per sentence (WordNet)
BACK_TRANS_BATCH = 8         # texts per nlpaug call
BACK_TRANS_LIMIT = None      # None = all PCL; int = cap at N samples (for speed)

GEMINI_MODEL = 'gemini-2.5-flash'
GEMINI_BATCH_SIZE = 50
GEMINI_TIMEOUT_MS = 180_000  # 3 min
GEMINI_TEMP = 0.8

if os.path.exists('/kaggle/input'):
    DATA_ROOT = '/kaggle/input/datasets/wowthecoder/patronizing-and-condescending-language-detection'
else:
    DATA_ROOT = '..'

if os.path.exists('/kaggle/working'):
    AUGMENTED_OUTPUT_CSV = '/kaggle/working/augmented_train_data.csv'
    AUGMENTED_DEV_OUTPUT_CSV = '/kaggle/working/augmented_dev_data.csv'
else:
    AUGMENTED_OUTPUT_CSV = 'augmented_train_data.csv'
    AUGMENTED_DEV_OUTPUT_CSV = 'augmented_dev_data.csv'

TSV_PATH = os.path.join(DATA_ROOT, 'dontpatronizeme_pcl.tsv')
TRAIN_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'train_semeval_parids-labels.csv')
DEV_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'dev_semeval_parids-labels.csv')

print(f'DATA_ROOT                 : {DATA_ROOT}')
print(f'AUGMENTED_OUTPUT_CSV      : {AUGMENTED_OUTPUT_CSV}')
print(f'AUGMENTED_DEV_OUTPUT_CSV  : {AUGMENTED_DEV_OUTPUT_CSV}')
print(f'AUGMENT_TRAIN_SPLIT       : {AUGMENT_TRAIN_SPLIT}')
print(f'AUGMENT_DEV_SPLIT         : {AUGMENT_DEV_SPLIT}')
print(f'New samples per original  : {TOTAL_NEW_PER_SAMPLE}')
print(
    'Breakdown                : '
    f'syn={SYNONYM_COPIES_PER_SAMPLE}, '
    f'bt={BACK_TRANS_COPIES_PER_SAMPLE}, '
    f'gemini={GEMINI_COPIES_PER_SAMPLE}, '
    f'dup={DUPLICATE_COPIES_PER_SAMPLE}'
)


DATA_ROOT                 : /kaggle/input/datasets/wowthecoder/patronizing-and-condescending-language-detection
AUGMENTED_OUTPUT_CSV      : /kaggle/working/augmented_train_data.csv
AUGMENTED_DEV_OUTPUT_CSV  : /kaggle/working/augmented_dev_data.csv
AUGMENT_TRAIN_SPLIT       : False
AUGMENT_DEV_SPLIT         : True
New samples per original  : 7
Breakdown                : syn=1, bt=2, gemini=2, dup=2


In [5]:
import contractions

def load_task1(train_path: str) -> pd.DataFrame:
    """
    Load Task 1 data and convert original labels to binary:
      0/1 -> 0 (No-PCL)
      2/3/4 -> 1 (PCL)
    """
    rows = []
    with open(train_path, encoding='utf-8') as f:
        for line in f.readlines()[4:]:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 6:
                continue

            par_id = parts[0]
            art_id = parts[1]
            keyword = parts[2]
            country = parts[3]
            text = parts[4]
            orig_label = parts[-1]
            label = 0 if orig_label in {'0', '1'} else 1

            rows.append(
                {
                    'par_id': str(par_id),
                    'art_id': art_id,
                    'keyword': keyword,
                    'country': country,
                    'text': text,
                    'label': label,
                    'orig_label': orig_label,
                }
            )

    return pd.DataFrame(
        rows,
        columns=['par_id', 'art_id', 'keyword', 'country', 'text', 'label', 'orig_label'],
    )


def preprocess_text(text: str) -> str:
    text = str(text)
    text = contractions.fix(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = load_task1(TSV_PATH)
df['clean_text'] = df['text'].apply(preprocess_text)

print(f'Loaded dataset: {len(df):,} samples')
print(df['label'].value_counts().sort_index().rename({0: 'No-PCL', 1: 'PCL'}))


Loaded dataset: 10,469 samples
label
No-PCL    9476
PCL        993
Name: count, dtype: int64


In [6]:
# ============================================================
# Official Train/Dev split
# ============================================================
train_ids_df = pd.read_csv(TRAIN_IDS_PATH, dtype={'par_id': str})
dev_ids_df = pd.read_csv(DEV_IDS_PATH, dtype={'par_id': str})

train_par_ids = set(train_ids_df['par_id'].astype(str))
dev_par_ids = set(dev_ids_df['par_id'].astype(str))

train_df = df[df['par_id'].isin(train_par_ids)].copy().reset_index(drop=True)
dev_df = df[df['par_id'].isin(dev_par_ids)].copy().reset_index(drop=True)

leftover_df = df[~df['par_id'].isin(train_par_ids | dev_par_ids)].copy().reset_index(drop=True)
if len(leftover_df) > 0:
    train_df = pd.concat([train_df, leftover_df], ignore_index=True)
    print(f'Appended {len(leftover_df):,} unassigned samples to training set.')


def describe_split(name: str, frame: pd.DataFrame):
    n = len(frame)
    n_pcl = int((frame['label'] == 1).sum())
    n_no_pcl = int((frame['label'] == 0).sum())
    ratio = f'1:{(n_no_pcl / n_pcl):.1f}' if n_pcl > 0 else 'undefined'
    print(f'{name:<8} -> total={n:,} | PCL={n_pcl:,} | No-PCL={n_no_pcl:,} | ratio={ratio}')


describe_split('Train(raw)', train_df)
describe_split('Dev(raw)', dev_df)


Train(raw) -> total=8,375 | PCL=794 | No-PCL=7,581 | ratio=1:9.5
Dev(raw) -> total=2,094 | PCL=199 | No-PCL=1,895 | ratio=1:9.5


In [7]:
# ============================================================
# Augmentation 1 - Synonym Replacement (WordNet)
# Shared train/dev augmentation helpers
# ============================================================
STOP_WORDS = set(nltk_stopwords.words('english'))


def get_wordnet_synonyms(word):
    """Return WordNet synonyms for word (single-token, different from original)."""
    syns = set()
    for synset in wordnet.synsets(word):
        for lemma in synset.lemmas():
            candidate = lemma.name().replace('_', ' ').lower()
            if candidate != word.lower() and ' ' not in candidate:
                syns.add(candidate)
    return list(syns)


def synonym_replacement(text, n_min=SYNONYM_N_MIN, n_max=SYNONYM_N_MAX):
    """Replace random content words with WordNet synonyms."""
    words = text.split()
    if not words:
        return text

    n = random.randint(n_min, min(n_max, len(words)))
    candidates = [
        i for i, w in enumerate(words)
        if w.isalpha() and w.lower() not in STOP_WORDS
    ]
    random.shuffle(candidates)

    new_words = words.copy()
    replaced = 0
    for idx in candidates:
        if replaced >= n:
            break
        syns = get_wordnet_synonyms(words[idx])
        if syns:
            new_words[idx] = random.choice(syns)
            replaced += 1
    return ' '.join(new_words)


def augment_with_synonyms(pcl_frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    for _, row in pcl_frame.iterrows():
        for copy_idx in range(SYNONYM_COPIES_PER_SAMPLE):
            aug_text = preprocess_text(synonym_replacement(row['clean_text']))
            new_row = row.copy()
            new_row['par_id'] = f"{row['par_id']}_syn{copy_idx + 1}"
            new_row['clean_text'] = aug_text
            rows.append(new_row)

    out = pd.DataFrame(rows).reset_index(drop=True)
    expected = len(pcl_frame) * SYNONYM_COPIES_PER_SAMPLE
    print(f'Synonym replacement ({split_name}): {len(out)} new PCL samples generated (expected {expected}).')
    return out


all_split_raw_frames = {
    'train': train_df,
    'dev': dev_df,
}
split_enabled = {
    'train': AUGMENT_TRAIN_SPLIT,
    'dev': AUGMENT_DEV_SPLIT,
}
split_raw_frames = {
    split_name: frame
    for split_name, frame in all_split_raw_frames.items()
    if split_enabled.get(split_name, False)
}
if not split_raw_frames:
    raise ValueError('No splits selected for augmentation. Enable AUGMENT_TRAIN_SPLIT and/or AUGMENT_DEV_SPLIT.')

print(f"Active augmentation splits: {', '.join(split_raw_frames.keys())}")

split_pcl_frames = {
    split_name: frame[frame['label'] == 1].copy().reset_index(drop=True)
    for split_name, frame in split_raw_frames.items()
}

for split_name, pcl_frame in split_pcl_frames.items():
    print(f'{split_name.capitalize()} PCL pool: {len(pcl_frame)}')

aug_syn_by_split = {
    split_name: augment_with_synonyms(pcl_frame, split_name)
    for split_name, pcl_frame in split_pcl_frames.items()
}

# Backward-compatible aliases used by later cells
empty_template_df = train_df.head(0).copy().reset_index(drop=True)

pcl_train = split_pcl_frames.get('train', empty_template_df.copy())
pcl_dev = split_pcl_frames.get('dev', empty_template_df.copy())

aug_syn_df = aug_syn_by_split.get('train', empty_template_df.copy())
aug_syn_dev_df = aug_syn_by_split.get('dev', empty_template_df.copy())


Active augmentation splits: dev
Dev PCL pool: 199
Synonym replacement (dev): 199 new PCL samples generated (expected 199).


In [8]:
# ============================================================
# Augmentation 2 - Gemini Paraphrasing (Gemini 2.5 Flash)
# Generates GEMINI_COPIES_PER_SAMPLE rows per original PCL sample.
# ============================================================
GEMINI_SYSTEM_PROMPT = """You are an expert linguist and data scientist specializing in identifying and generating Patronizing and Condescending Language (PCL).

An entity engages in PCL when its language use shows a superior attitude towards others or depicts them in a compassionate way. This effect is not always conscious and the intention of the author is often to help the person or group they refer to (e.g. by raising awareness or funds, or moving the audience to action). However, these superior attitudes and a discourse of pity can routinize discrimination and make it less visible.

Your task is to paraphrase sentences to augment a machine learning dataset. You MUST strictly adhere to these rules:
1. PRESERVE THE TONE: You must maintain the exact same patronizing tone, "savior complex", pity, or implicit superiority found in the original text if the label indicates that the text contains PCL.
2. DO NOT SANITIZE: Do not neutralize the text into polite, objective English.
3. NO OVERT TOXICITY: PCL is subtle. Do not use slurs, aggressive hate speech, or overt hostility if not present in the original text.
4. NO CHATTER: Output ONLY the paraphrased sentence. Do not include quotes or explanations."""


def parse_gemini_response(response_text, expected_count):
    """Extract numbered list items from Gemini response."""
    paraphrases = {}
    for line in response_text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        m = re.match(r'^(\d+)[.):]\s*(.+)$', line)
        if m:
            idx = int(m.group(1))
            text = m.group(2).strip()
            if (text.startswith('"') and text.endswith('"')) or (text.startswith("'") and text.endswith("'")):
                text = text[1:-1].strip()
            if 1 <= idx <= expected_count and text:
                paraphrases[idx] = text
    return paraphrases


def gemini_paraphrase_batch(client, texts):
    """Paraphrase a batch of texts. Falls back to originals on error."""
    n = len(texts)
    numbered = '\n'.join(f'{i + 1}. {t}' for i, t in enumerate(texts))
    user_prompt = (
        f'Paraphrase each of the following {n} sentences. '
        f'Output ONLY a numbered list in the exact same format (number. text), '
        f'one paraphrase per line, no extra commentary:\n\n{numbered}'
    )
    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=user_prompt,
            config=genai_types.GenerateContentConfig(
                system_instruction=GEMINI_SYSTEM_PROMPT,
                temperature=GEMINI_TEMP,
                http_options=genai_types.HttpOptions(timeout=GEMINI_TIMEOUT_MS),
            ),
        )
        parsed = parse_gemini_response(response.text, n)
        return [preprocess_text(parsed.get(i + 1, texts[i])) for i in range(n)]
    except Exception as e:
        print(f'  Gemini batch failed: {e}')
        return texts


def augment_with_gemini(pcl_frame: pd.DataFrame, split_name: str, client, ready: bool) -> pd.DataFrame:
    rows = []
    source_texts = pcl_frame['clean_text'].tolist()

    for copy_idx in range(GEMINI_COPIES_PER_SAMPLE):
        all_paraphrases = []

        if ready:
            for start in range(0, len(source_texts), GEMINI_BATCH_SIZE):
                batch_texts = source_texts[start : start + GEMINI_BATCH_SIZE]
                batch_paras = gemini_paraphrase_batch(client, batch_texts)
                all_paraphrases.extend(batch_paras)
                done = min(start + GEMINI_BATCH_SIZE, len(source_texts))
                print(f'  {split_name} round {copy_idx + 1}: {done}/{len(source_texts)} paraphrased')
                time.sleep(1)
        else:
            all_paraphrases = source_texts

        for row, para in zip(pcl_frame.itertuples(index=False), all_paraphrases):
            rows.append({
                'par_id':     f"{row.par_id}_gemini{copy_idx + 1}",
                'art_id':     row.art_id,
                'keyword':    row.keyword,
                'country':    row.country,
                'text':       row.text,
                'clean_text': preprocess_text(para),
                'label':      1,
                'orig_label': row.orig_label,
            })

    out = pd.DataFrame(rows).reset_index(drop=True)
    expected = len(pcl_frame) * GEMINI_COPIES_PER_SAMPLE
    print(f'Gemini paraphrasing ({split_name}): {len(out)} new PCL samples generated (expected {expected}).')
    return out


try:
    gemini_client = genai.Client(
        api_key=GEMINI_API_KEY,
        http_options=genai_types.HttpOptions(
            timeout=GEMINI_TIMEOUT_MS,
            retry_options=genai_types.HttpRetryOptions(
                attempts=6,
                initial_delay=1.0,
                max_delay=30.0,
                exp_base=2.0,
                jitter=1.0,
                http_status_codes=[408, 429, 500, 502, 503, 504],
            ),
        ),
    )
    gemini_ready = True
    for split_name, pcl_frame in split_pcl_frames.items():
        print(f'Paraphrasing {len(pcl_frame)} {split_name} PCL samples with {GEMINI_MODEL} ({GEMINI_BATCH_SIZE} per request) ...')
except Exception as e:
    print(f'Gemini client setup failed: {e}')
    print('Using original texts as Gemini fallbacks.')
    gemini_ready = False
    gemini_client = None

aug_gemini_by_split = {
    split_name: augment_with_gemini(pcl_frame, split_name, gemini_client, gemini_ready)
    for split_name, pcl_frame in split_pcl_frames.items()
}

# Backward-compatible aliases used by later cells
aug_gemini_df = aug_gemini_by_split.get('train', empty_template_df.copy())
aug_gemini_dev_df = aug_gemini_by_split.get('dev', empty_template_df.copy())


Paraphrasing 199 dev PCL samples with gemini-2.5-flash (50 per request) ...
  dev round 1: 50/199 paraphrased
  dev round 1: 100/199 paraphrased
  dev round 1: 150/199 paraphrased
  dev round 1: 199/199 paraphrased
  dev round 2: 50/199 paraphrased
  dev round 2: 100/199 paraphrased
  dev round 2: 150/199 paraphrased
  dev round 2: 199/199 paraphrased
Gemini paraphrasing (dev): 398 new PCL samples generated (expected 398).


In [9]:
# ============================================================
# Augmentation 3 - Back Translation (nlpaug + Helsinki OPUS-MT)
# Generates BACK_TRANS_COPIES_PER_SAMPLE rows per source sample.
# ============================================================
import math
from tqdm import tqdm


def run_back_translation_once(augmenter, input_texts, copy_idx, total_copies):
    """Run one EN->DE->EN pass over all input_texts and return cleaned outputs."""
    out = []
    total_batches = math.ceil(len(input_texts) / BACK_TRANS_BATCH) if input_texts else 0
    print(
        f'Starting BT copy {copy_idx + 1}/{total_copies} '
        f'({len(input_texts)} texts, {total_batches} batches)'
    )

    for batch_idx, i in enumerate(
        tqdm(
            range(0, len(input_texts), BACK_TRANS_BATCH),
            total=total_batches,
            desc=f'BT copy {copy_idx + 1}',
        ),
        start=1,
    ):
        batch = input_texts[i : i + BACK_TRANS_BATCH]
        try:
            augmented = augmenter.augment(batch)
            if augmented and isinstance(augmented[0], list):
                augmented = [a[0] if a else b for a, b in zip(augmented, batch)]
        except Exception as e:
            print(f'  Batch {batch_idx}/{total_batches} failed: {e} - using originals', flush=True)
            augmented = batch

        if len(augmented) != len(batch):
            augmented = batch

        for bt_text in augmented:
            out.append(preprocess_text(bt_text))

        if batch_idx == 1 or batch_idx % 5 == 0 or batch_idx == total_batches:
            done = min(i + BACK_TRANS_BATCH, len(input_texts))
            print(f'  Progress: {done}/{len(input_texts)} texts', flush=True)

    return out


def augment_with_back_translation(
    pcl_frame: pd.DataFrame,
    split_name: str,
    augmenter,
    ready: bool,
    limit=None,
):
    source_pool = pcl_frame if limit is None else pcl_frame.head(limit)
    source_texts = source_pool['clean_text'].tolist()
    rows = []

    for copy_idx in range(BACK_TRANS_COPIES_PER_SAMPLE):
        if ready:
            bt_texts = run_back_translation_once(augmenter, source_texts, copy_idx, BACK_TRANS_COPIES_PER_SAMPLE)
        else:
            bt_texts = source_texts

        for (_, row), bt_text in zip(source_pool.iterrows(), bt_texts):
            new_row = row.copy()
            new_row['par_id'] = f"{row['par_id']}_bt{copy_idx + 1}"
            new_row['clean_text'] = bt_text
            rows.append(new_row)

    out = pd.DataFrame(rows).reset_index(drop=True)
    expected = len(source_pool) * BACK_TRANS_COPIES_PER_SAMPLE
    print(f'Back translation ({split_name}): {len(out)} new PCL samples generated (expected {expected}).')
    return out, source_pool


try:
    bt_aug = naw.BackTranslationAug(
        from_model_name='Helsinki-NLP/opus-mt-en-de',
        to_model_name='Helsinki-NLP/opus-mt-de-en',
        device='cuda' if torch.cuda.is_available() else 'cpu',
        max_length=512,
        batch_size=BACK_TRANS_BATCH,
    )
    print('Back-translation augmenter ready.')
    bt_ready = True
except Exception as e:
    print(f'Back translation setup failed: {e}')
    print('Using original texts as BT fallbacks.')
    bt_ready = False
    bt_aug = None

aug_bt_by_split = {}
bt_pool_by_split = {}
for split_name, pcl_frame in split_pcl_frames.items():
    aug_bt_by_split[split_name], bt_pool_by_split[split_name] = augment_with_back_translation(
        pcl_frame,
        split_name,
        bt_aug,
        bt_ready,
        limit=BACK_TRANS_LIMIT,
    )

# Backward-compatible aliases used by later cells
aug_bt_df = aug_bt_by_split.get('train', empty_template_df.copy())
aug_bt_dev_df = aug_bt_by_split.get('dev', empty_template_df.copy())
pcl_bt_pool = bt_pool_by_split.get('train', empty_template_df.copy())
pcl_bt_dev_pool = bt_pool_by_split.get('dev', empty_template_df.copy())


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

...

  Progress: 160/199 texts


BT copy 2:  96%|█████████▌| 24/25 [00:59<00:02,  2.34s/it]

  Progress: 199/199 texts


BT copy 2: 100%|██████████| 25/25 [01:01<00:00,  2.46s/it]

Back translation (dev): 398 new PCL samples generated (expected 398).


In [10]:
# ============================================================
# Optional export - Back-translation-only samples
# Run this after Augmentation 3 if you only want BT outputs.
# ============================================================
from pathlib import Path

bt_export_paths = {
    'train': Path('back_translated_samples.csv'),
    'dev': Path('back_translated_dev_samples.csv'),
}

for split_name in split_raw_frames:
    out_path = bt_export_paths[split_name]
    bt_frame = aug_bt_by_split.get(split_name, empty_template_df.copy())
    bt_frame.to_csv(out_path, index=False)
    print(f'Saved {split_name} back-translation-only CSV: {out_path.resolve()} (rows={len(bt_frame)})')


Saved dev back-translation-only CSV: /kaggle/working/back_translated_dev_samples.csv (rows=398)


In [11]:
# ============================================================
# Augmentation 4 - Duplication + Combine + CSV Export
# Shared combine/export flow for train and dev
# ============================================================

def augment_with_duplication(pcl_frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    for _, row in pcl_frame.iterrows():
        for copy_idx in range(DUPLICATE_COPIES_PER_SAMPLE):
            new_row = row.copy()
            new_row['par_id'] = f"{row['par_id']}_dup{copy_idx + 1}"
            rows.append(new_row)

    out = pd.DataFrame(rows).reset_index(drop=True)
    expected = len(pcl_frame) * DUPLICATE_COPIES_PER_SAMPLE
    print(f'Duplication ({split_name}): {len(out)} new PCL samples generated (expected {expected}).')
    return out


def to_augmented_only(split_final_df: pd.DataFrame, split_raw_df: pd.DataFrame) -> pd.DataFrame:
    orig_ids = set(split_raw_df['par_id'].astype(str))
    out = split_final_df[~split_final_df['par_id'].astype(str).isin(orig_ids)].copy()

    cols = ['par_id', 'art_id', 'keyword', 'country', 'text', 'clean_text', 'label', 'orig_label']
    for col in cols:
        if col not in out.columns:
            out[col] = ''
    return out[cols].reset_index(drop=True)


def describe(name, frame):
    n = len(frame)
    n_pcl = int((frame['label'] == 1).sum()) if n else 0
    n_no = int((frame['label'] == 0).sum()) if n else 0
    frac = (n_pcl / n) if n > 0 else 0.0
    print(f'{name:<22} -> total={n:,} | PCL={n_pcl:,} ({frac:.1%}) | No-PCL={n_no:,}')


aug_dup_by_split = {
    split_name: augment_with_duplication(pcl_frame, split_name)
    for split_name, pcl_frame in split_pcl_frames.items()
}

# Keep parity with the current notebook behavior: final CSV excludes BT rows.
method_frames_by_split = {
    split_name: [
        aug_syn_by_split[split_name],
        aug_gemini_by_split[split_name],
        aug_dup_by_split[split_name],
    ]
    for split_name in split_raw_frames
}

output_paths = {
    'train': AUGMENTED_OUTPUT_CSV,
    'dev': AUGMENTED_DEV_OUTPUT_CSV,
}

split_results = {}
for split_name, raw_frame in split_raw_frames.items():
    merged = pd.concat([raw_frame, *method_frames_by_split[split_name]], ignore_index=True)
    merged['label'] = merged['label'].astype(int)
    final_df = merged.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    augmented_only_df = to_augmented_only(final_df, raw_frame)

    out_path = output_paths[split_name]
    augmented_only_df.to_csv(out_path, index=False)

    split_results[split_name] = {
        'merged': merged,
        'final': final_df,
        'aug_only': augmented_only_df,
        'path': out_path,
    }

    expected_syn = len(split_pcl_frames[split_name]) * SYNONYM_COPIES_PER_SAMPLE
    expected_bt = len(bt_pool_by_split[split_name]) * BACK_TRANS_COPIES_PER_SAMPLE
    expected_gem = len(split_pcl_frames[split_name]) * GEMINI_COPIES_PER_SAMPLE
    expected_dup = len(split_pcl_frames[split_name]) * DUPLICATE_COPIES_PER_SAMPLE
    expected_aug_total = expected_syn + expected_gem + expected_dup

    print(f'\n[{split_name.upper()}] Expected/actual generated rows:')
    print(f'  synonym      : expected={expected_syn:,}, actual={len(aug_syn_by_split[split_name]):,}')
    print(f'  back-trans   : expected={expected_bt:,}, actual={len(aug_bt_by_split[split_name]):,}')
    print(f'  gemini       : expected={expected_gem:,}, actual={len(aug_gemini_by_split[split_name]):,}')
    print(f'  duplication  : expected={expected_dup:,}, actual={len(aug_dup_by_split[split_name]):,}')
    print(f'  total aug    : expected={expected_aug_total:,}, actual={len(augmented_only_df):,}')

    if BACK_TRANS_LIMIT is None:
        expected_for_7x = len(split_pcl_frames[split_name]) * TOTAL_NEW_PER_SAMPLE
        print(f'  7x target check (all methods on all PCL rows): expected={expected_for_7x:,}')

    describe(f'{split_name.capitalize()}(raw)', raw_frame)
    describe(f'{split_name.capitalize()}(aug rows)', augmented_only_df)
    describe(f'{split_name.capitalize()}(final)', final_df)
    print(f'Saved {split_name} augmented CSV: {out_path}')

# Backward-compatible aliases
train_aug_df = split_results.get('train', {}).get('merged', train_df.copy())
train_final_df = split_results.get('train', {}).get('final', train_df.copy())
augmented_final_df = split_results.get('train', {}).get('aug_only', empty_template_df.copy())

dev_aug_df = split_results.get('dev', {}).get('merged', dev_df.copy())
dev_final_df = split_results.get('dev', {}).get('final', dev_df.copy())
augmented_dev_final_df = split_results.get('dev', {}).get('aug_only', empty_template_df.copy())


Duplication (dev): 398 new PCL samples generated (expected 398).

[DEV] Expected/actual generated rows:
  synonym      : expected=199, actual=199
  back-trans   : expected=398, actual=398
  gemini       : expected=398, actual=398
  duplication  : expected=398, actual=398
  total aug    : expected=995, actual=995
  7x target check (all methods on all PCL rows): expected=1,393
Dev(raw)               -> total=2,094 | PCL=199 (9.5%) | No-PCL=1,895
Dev(aug rows)          -> total=995 | PCL=995 (100.0%) | No-PCL=0
Dev(final)             -> total=3,089 | PCL=1,194 (38.7%) | No-PCL=1,895
Saved dev augmented CSV: /kaggle/working/augmented_dev_data.csv
